# BitFit 实战

## Step1 导入相关包

In [ ]:
import os
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForSeq2Seq,
    TrainingArguments,
    Trainer,
)

## Step2 加载数据集

In [ ]:
ds = Dataset.load_from_disk("./data/alpaca_data_zh/")
ds

In [ ]:
ds[:3]

## Step3 数据集预处理

In [ ]:
help(AutoTokenizer.from_pretrained)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("./models/Langboat/bloom-1b4-zh")
tokenizer

In [ ]:
def process_func(example):
    # print(f"example: {example}")

    MAX_LENGTH = 256
    input_ids, attention_mask, labels = [], [], []

    # print(["Human: " + example["instruction"], example["input"]])
    # print("------------")
    # print("\n".join(["Human: " + example["instruction"], example["input"]]).strip())
    # print("------------")
    # print("\n".join(["Human: " + example["instruction"], example["input"]]).strip() + "\n\nAssistant: ")
    # print("------------")
    # print(example["output"] + tokenizer.eos_token)
    # print("------------" * 10)

    instruction = tokenizer(
        "\n".join(["Human: " + example["instruction"], example["input"]]).strip()
        + "\n\nAssistant: "
    )

    response = tokenizer(example["output"] + tokenizer.eos_token)

    input_ids = instruction["input_ids"] + response["input_ids"]

    attention_mask = instruction["attention_mask"] + response["attention_mask"]

    labels = [-100] * len(instruction["input_ids"]) + response["input_ids"]

    if len(input_ids) > MAX_LENGTH:
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]

    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

In [ ]:
# 调试
# ds = ds.select([0, 3, 1])
# ds

In [ ]:
# 调试
# tokenized_ds = ds.map(process_func, remove_columns=ds.column_names)
# tokenized_ds

In [ ]:
# 调试
# tokenized_ds[1]

In [ ]:
# 调试
# tokenizer.decode(tokenized_ds[1]["input_ids"])

In [ ]:
# 调试
# tokenizer.decode(
#     list(
#         filter(lambda x: x != -100, tokenized_ds[1]["labels"])
#     )
# )

In [ ]:
tokenized_ds = ds.map(process_func, remove_columns=ds.column_names)
tokenized_ds

In [ ]:
tokenizer.decode(tokenized_ds[1]["input_ids"])

In [ ]:
tokenizer.decode(list(filter(lambda x: x != -100, tokenized_ds[1]["labels"])))

## Step4 创建模型

In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0, 1, 2, 3, 4, 5, 6, 7"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
help(AutoModelForCausalLM.from_pretrained)

In [ ]:
# model = AutoModelForCausalLM.from_pretrained("./models/Langboat/bloom-1b4-zh", device_map=0)
model = AutoModelForCausalLM.from_pretrained("./models/Langboat/bloom-1b4-zh", device_map="auto")

In [ ]:
model.device

In [ ]:
model.config

In [ ]:
# 计算模型的参数
sum(param.numel() for param in model.parameters())

model size: 1.3B

model: 1.3G * 4 ~= 5.2G

gradient: 1.3G * 4 ~= 5.2G

optimizer: 1.3G * 4 * 2 ~= 10.4G

sum: 20.8G

## BitFit

In [ ]:
# bitfit
# 选择模型参数里面的所有 bias 部分

num_param = 0
for name, param in model.named_parameters():
    print(f"name: {name}")
    if "bias" not in name:
        param.requires_grad = False
    else:
        num_param += param.numel()

num_param

In [ ]:
num_param / sum(param.numel() for param in model.parameters())

## Step5 配置训练参数

In [ ]:
args = TrainingArguments(
    output_dir="./chatbot",
    per_device_train_batch_size=32,
    gradient_accumulation_steps=32,
    logging_steps=10,
    num_train_epochs=1,
    report_to="none"
)

## Step6 创建训练器

In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    tokenizer=tokenizer,
    train_dataset=tokenized_ds,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
)

## Step7 模型训练

In [ ]:
trainer.train()

In [ ]:
model = model.cuda()
ipt = tokenizer(
    "Human: {}\n{}".format("考试有哪些技巧？", "").strip() + "\n\nAssistant: ",
    return_tensors="pt",
).to(model.device)
tokenizer.decode(
    model.generate(**ipt, max_length=128, do_sample=True)[0], skip_special_tokens=True
)

## Step8 模型推理

In [ ]:
from transformers import pipeline

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, device=0)

In [ ]:
ipt = "Human: {}\n{}".format("考试有哪些技巧？", "").strip() + "\n\nAssistant: "
pipe(
    ipt,
    max_length=256,
    do_sample=True,
)